In [1]:
# Para evitar el overfitting del primer intento en que el usábamos nivel de S2 = 12 y MIN_FOTOS = 3
# Vamos a realizar un pequeño barrido para determinar cual puede ser la configuración ideal para nuestro modelo

import os
import sys

# Forzamos la caché en el disco grande ANTES de importar datasets
CACHE_DIR = "D:/Datasets_Cache"
os.environ["HF_DATASETS_CACHE"] = CACHE_DIR
os.makedirs(CACHE_DIR, exist_ok=True)

import s2sphere
import collections
import time
import pandas as pd
import multiprocessing
from datasets import load_dataset

# Configuración del barrido
NIVELES_S2 = [9, 10, 11, 12]
UMBRALES_IMAGENES = [3, 20, 30, 50, 100, 200]
NUM_CORES = max(1, multiprocessing.cpu_count() - 1)

def batch_process_s2_sweep(batch, level):
    """Calcula los tokens S2 dinámicamente según el nivel pasado por parámetro"""
    import s2sphere
    tokens = []
    for lat, lon in zip(batch['latitude'], batch['longitude']):
        token = None
        try:
            if lat is not None and lon is not None:
                if not (abs(lat) > 90 or abs(lon) > 180 or (lat == 0 and lon == 0)):
                    p1 = s2sphere.LatLng.from_degrees(lat, lon)
                    token = s2sphere.CellId.from_lat_lng(p1).parent(level).to_token()
        except:
            pass
        tokens.append(token)
    return {"s2_token": tokens}

print("Cargando OSV-5M desde caché")
dataset = load_dataset(
    "osv5m/osv5m", 
    split="train", 
    streaming=False, 
    trust_remote_code=True,
    keep_in_memory=False,
    cache_dir=CACHE_DIR
)
# Nos quedamos solo con lat y lon para máxima eficiencia en RAM
dataset = dataset.remove_columns([c for c in dataset.column_names if c not in {'latitude', 'longitude'}])
print(f"Dataset cargado: {len(dataset):,} filas.\n")

resultados = []

# Empezamos el barrido
for nivel in NIVELES_S2:
    print(f"Iniciando procesamiento para Nivel S2: {nivel}")
    start_time = time.time()
    
    # Procesamiento en paralelo
    ds_nivel = dataset.map(
        batch_process_s2_sweep,
        fn_kwargs={"level": nivel},                # Pasamos el nivel a la función
        batched=True,
        batch_size=5000,
        num_proc=NUM_CORES,
        remove_columns=['latitude', 'longitude'],  # Liberamos memoria
        desc=f"Tokens Nivel {nivel}"
    )
    
    # Extraemos tokens válidos y los contamos
    tokens_validos = [t for t in ds_nivel['s2_token'] if t is not None]
    counter = collections.Counter(tokens_validos)
    
    print(f"Procesado en {time.time() - start_time:.2f} segundos.")
    print(f"Celdas únicas encontradas a nivel {nivel}: {len(counter):,}\n")
    
    # Comprobamos los distintos umbrales de imágenes por clase
    for minimo_img in UMBRALES_IMAGENES:
        clases_validas = sum(1 for count in counter.values() if count >= minimo_img)
        resultados.append({
            "Nivel S2": nivel,
            "Mínimo Imágenes": minimo_img,
            "Clases Válidas (Reales)": clases_validas
        })

# Mostramos la tabla comparativa
df_resultados = pd.DataFrame(resultados)
# Aplicamos un formato bonito para visualizar mejor los números grandes
df_resultados["Clases Válidas (Reales)"] = df_resultados["Clases Válidas (Reales)"].apply(lambda x: f"{x:,}")
display(df_resultados)

d:\Projects\GeoGuessrAI\venv\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Cargando OSV-5M desde caché
OSV5M {'full': False, 'name': 'osv5m', 'hash': '0ac7fc681aace0f00245c6dea7848bcb0e424845081905154c1adb5f9bf8f19e', 'base_path': 'https://huggingface.co/datasets/osv5m/osv5m/resolve/main', 'token': None, 'use_auth_token': None, 'repo_id': 'osv5m/osv5m', 'storage_options': {}, 'dataset_name': 'osv5m', '_writer_batch_size': None, 'config_kwargs': {}, 'config': BuilderConfig(name='default', version=0.0.0, data_dir=None, data_files=None, description=None), 'config_id': 'default', 'info': DatasetInfo(description='', citation='', homepage='', license='', features={'image': Image(mode=None, decode=True, id=None), 'latitude': Value(dtype='float32', id=None), 'longitude': Value(dtype='float32', id=None), 'country': Value(dtype='string', id=None), 'region': Value(dtype='string', id=None), 'sub-region': Value(dtype='string', id=None), 'city': Value(dtype='string', id=None)}, post_processed=None, supervised_keys=None, task_templates=None, builder_name='osv5m', dataset_na

,Nivel S2,Mínimo Imágenes,Clases Válidas (Reales)
0,9,3,"51,026"
1,9,20,"34,720"
2,9,30,"29,697"
3,9,50,"22,583"
4,9,100,"12,913"
5,9,200,"5,763"
6,10,3,"115,191"
7,10,20,"60,862"
8,10,30,"45,009"
9,10,50,"26,361"


In [2]:
# Teniendo en cuenta los resultados de la tabla superior vamos a escoger:
# Nivel 10 (celdas de 50-100km) Mínimo 100 imágenes -> Unas 10 mil clases
# Nos permite una configuración que cubra áreas geográficas grandes con un nivel adecuado de imágenes adecuado para que el modelo aprenda

# Extra: Una alternativa agresiva que se podría probar es Nivel 11 (celdas de 15-25km) tratando de atinar más al detalle
# Si sobra tiempo es modelo interesante a probar


# Este cuaderno se encarga de:
# 1. Descargar o cargar desde cache el dataset OSV-5M
# 2. Convierte las coordenadas de cada imagen a un "Token S2" del nivel elegido
#    S2 es un sistema de Google que divide la esfera terrestre en celdas de tamaño variable
# 3. Filtrado de clases, las celdas que tengan menos imágenes que el mínimo se eliminan
# 4. Genera un diccionario ("s2_class_map.pkl") con la información de cada celda válida
# 5. Genera un mapa visual ("mapa_cobertura.html") con los centros de las celdas para poder visualizarlos
# 6. Genera una lista de índices ("train_indices.pkl") con todas las imágenes que pertenecen a las clases válidas
# 7. Genera también la lista de índices ("val_indices.pkl")

import os
import sys
import multiprocessing

CACHE_DIR = "D:/Datasets_Cache"                                 # Ruta donde guardar el dataset
MODELS_DIR = "../models"                                        # Carpeta donde guardar el output del notebook
OUTPUT_FILE = os.path.join(MODELS_DIR,"s2_class_map.pkl")       # Archivo final de clases
MAP_FILENAME = os.path.join(MODELS_DIR,"mapa_cobertura.html")   # Archivo visual con los centros de las celdas
S2_LEVEL = 10                                                   # Nivel de precisión o tamaño de celda (~50-100km cuadrados con nivel = 10)
MIN_IMAGES_PER_CLASS = 100                                      # Mínimo de fotos para aceptar una celda
NUM_CORES = max(1, multiprocessing.cpu_count() - 1)             # Dejar al menos 1 libre para el SO

# Forzamos la caché en el disco grande ANTES de importar datasets
os.environ["HF_DATASETS_CACHE"] = CACHE_DIR
os.makedirs(CACHE_DIR, exist_ok=True)

# Instalación automática de dependencias
print("Verificando librerías...")
os.system('pip install -q "datasets<3.0.0" s2sphere folium')
print(f"Configuración lista. Usando caché en: {CACHE_DIR}")
print(f"Núcleos disponibles para procesamiento: {NUM_CORES}")

Verificando librerías...
Configuración lista. Usando caché en: D:/Datasets_Cache
Núcleos disponibles para procesamiento: 7


In [3]:
import s2sphere
import collections
import pickle
import random
import folium

# Esta función se encarga de convertir coordenadas a tokens S2 
# Optimizada para ejecutarse en paralelo (por eso el import local)
def batch_process_s2(batch):
    import s2sphere
    
    # Constante local para los workers
    LEVEL = 12 
    
    tokens = []
    lats = batch['latitude']
    lons = batch['longitude']
    
    for lat, lon in zip(lats, lons):
        token = None 
        try:
            # Validación de coordenadas
            if lat is not None and lon is not None:
                # Evitar el (0,0) y rangos imposibles
                if not (abs(lat) > 90 or abs(lon) > 180 or (lat == 0 and lon == 0)):
                    p1 = s2sphere.LatLng.from_degrees(lat, lon)
                    token = s2sphere.CellId.from_lat_lng(p1).parent(LEVEL).to_token()
        except Exception:
            pass # Si falla, se mantiene como None
            
        tokens.append(token)
            
    return {"s2_token": tokens}

# Función para validar la configuración de S2, si falla se detiene el proceso
def validate_s2_setup(dataset):
    print("\nRealizando prueba de diagnóstico...")
    try:
        sample = dataset[0]
        lat, lon = sample['latitude'], sample['longitude']
        p1 = s2sphere.LatLng.from_degrees(lat, lon)
        token = s2sphere.CellId.from_lat_lng(p1).parent(12).to_token()
        print(f"Prueba exitosa. Lat: {lat:.4f}, Lon: {lon:.4f} -> Token: {token}")
        return True
    except Exception as e:
        print(f"\nError crítico al validar S2: {e}")
        return False

In [4]:
import time
from datasets import load_dataset

def run_etl():
    print(f"Cargando OSV-5M desde caché local...")
    
    try:
        # Carga optimizada sin streaming (en local)
        dataset = load_dataset(
            "osv5m/osv5m", 
            split="train", 
            streaming=False,                # Modo de carga, estando en false primero descarga todo el dataset y luego empieza a trabajar
            trust_remote_code=True,         # Autoriza a cargar el dataset con el script de carga de Hugging Face
            keep_in_memory=False            # No mantiene todo el dataset en memoria ram (para evitar que colapse)
        )
        
        # Mantenemos solo las columnas de latitud y longitud, ahorrando memoria
        cols_to_keep = {'latitude', 'longitude'}
        cols_to_drop = [c for c in dataset.column_names if c not in cols_to_keep]
        dataset = dataset.remove_columns(cols_to_drop)
        print(f"Dataset listo: {len(dataset):,} filas.")
        
    except Exception as e:
        print(f"Error cargando dataset: {e}")
        return None

    # Verificación antes de empezar a procesar
    if not validate_s2_setup(dataset): return

    print(f"\nIniciando procesamiento en paralelo con {NUM_CORES} núcleos...")
    start_time = time.time()

    results = dataset.map(
        batch_process_s2,
        batched=True,
        batch_size=5000,                            # Cantidad de coordenadas por batch
        num_proc=NUM_CORES,
        remove_columns=['latitude', 'longitude'],   # Una vez calculado el token, no las necesitamos
        desc="Generando Tokens S2"
    )

    print("Agregando frecuencias...")
    # Filtramos los None
    all_tokens = [t for t in results['s2_token'] if t is not None]
    
    counter = collections.Counter(all_tokens)
    elapsed = time.time() - start_time
    print(f"Procesado en {elapsed:.2f} segundos.")

    print("Guardando archivo de clases...")
    valid_cells = [cell for cell, count in counter.items() if count >= MIN_IMAGES_PER_CLASS]
    valid_cells.sort(key=lambda x: counter[x], reverse=True)
    
    # Índices para la IA (Token -> ID numérico y viceversa)
    cell_to_idx = {cell: idx for idx, cell in enumerate(valid_cells)}
    idx_to_cell = {idx: cell for cell, idx in cell_to_idx.items()}
    
    final_data = {
        "cell_to_idx": cell_to_idx,
        "idx_to_cell": idx_to_cell,
        "total_cells": len(valid_cells),
        "s2_level": S2_LEVEL,
        "raw_counts": dict(counter)
    }
    
    with open(OUTPUT_FILE, "wb") as f:
        pickle.dump(final_data, f)
        
    print(f"Clases generadas: {len(valid_cells):,}")
    print(f"Archivo guardado: {os.path.abspath(OUTPUT_FILE)}")
    return OUTPUT_FILE, results

# Ejecutar
output_pkl, ds_with_tokens = run_etl()

Cargando OSV-5M desde caché local...
OSV5M {'full': False, 'name': 'osv5m', 'hash': '0ac7fc681aace0f00245c6dea7848bcb0e424845081905154c1adb5f9bf8f19e', 'base_path': 'https://huggingface.co/datasets/osv5m/osv5m/resolve/main', 'token': None, 'use_auth_token': None, 'repo_id': 'osv5m/osv5m', 'storage_options': {}, 'dataset_name': 'osv5m', '_writer_batch_size': None, 'config_kwargs': {}, 'config': BuilderConfig(name='default', version=0.0.0, data_dir=None, data_files=None, description=None), 'config_id': 'default', 'info': DatasetInfo(description='', citation='', homepage='', license='', features={'image': Image(mode=None, decode=True, id=None), 'latitude': Value(dtype='float32', id=None), 'longitude': Value(dtype='float32', id=None), 'country': Value(dtype='string', id=None), 'region': Value(dtype='string', id=None), 'sub-region': Value(dtype='string', id=None), 'city': Value(dtype='string', id=None)}, post_processed=None, supervised_keys=None, task_templates=None, builder_name='osv5m', d

In [5]:
def generate_map(pkl_file, html_file, samples=400000):
    if not pkl_file or not os.path.exists(pkl_file):
        print("No hay archivo PKL para visualizar")
        return

    print(f"\nGenerando mapa interactivo ({samples} puntos)")
    with open(pkl_file, "rb") as f:
        data = pickle.load(f)
        
    all_cells = list(data["cell_to_idx"].keys())
    
    # Muestreo aleatorio si hay demasiados puntos
    sampled_cells = random.sample(all_cells, min(samples, len(all_cells)))
    
    # Mapa base oscuro
    m = folium.Map(location=[20, 0], zoom_start=2, tiles='CartoDB dark_matter')
    
    count = 0
    for token in sampled_cells:
        try:
            cell_id = s2sphere.CellId.from_token(token)
            ll = cell_id.to_lat_lng()
            folium.CircleMarker(
                location=[ll.lat().degrees, ll.lng().degrees],
                radius=1,
                color='#33FF57', # Verde Matrix
                fill=True,
                fill_opacity=0.6,
                weight=0
            ).add_to(m)
            count += 1
        except:
            continue
            
    m.save(html_file)
    print(f"Mapa guardado en: {os.path.abspath(html_file)}")
    print("Ábrelo en tu navegador para ver la cobertura mundial.")

# Ejecutar
generate_map(OUTPUT_FILE, MAP_FILENAME)


Generando mapa interactivo (400000 puntos)
Mapa guardado en: d:\Projects\GeoGuessrAI\models\mapa_cobertura.html
Ábrelo en tu navegador para ver la cobertura mundial.


In [6]:
# Generador de índices de entrenamiento
from tqdm import tqdm

# Cargar el mapa de clases que acabamos de generar
print(f"Cargando mapa de clases desde {OUTPUT_FILE}")
with open(OUTPUT_FILE, "rb") as f:
    class_data = pickle.load(f)

# Valid cells son las claves del diccionario
valid_cells = set(class_data["cell_to_idx"].keys())
print(f"Total de celdas válidas: {len(valid_cells):,}")

print("Filtrando imágenes válidas desde los tokens en memoria")
train_indices = []

# Iteramos directamente sobre la lista de tokens que ya calculamos
for idx, token in enumerate(tqdm(ds_with_tokens['s2_token'], desc="Seleccionando imágenes")):
    if token in valid_cells:
        train_indices.append(idx)

# Guardar el archivo final
INDICES_FILE = os.path.join(MODELS_DIR, "train_indices.pkl")
with open(INDICES_FILE, "wb") as f:
    pickle.dump(train_indices, f)

print(f"\n¡Éxito! Indices guardados en: {os.path.abspath(INDICES_FILE)}")
print(f"Imágenes válidas para entrenar: {len(train_indices):,} (Se descartaron {len(ds_with_tokens) - len(train_indices):,})")

Cargando mapa de clases desde ../models\s2_class_map.pkl
Total de celdas válidas: 1,084
Filtrando imágenes válidas desde los tokens en memoria


Seleccionando imágenes: 100%|██████████| 4893782/4893782 [00:01<00:00, 2854265.31it/s]



¡Éxito! Indices guardados en: d:\Projects\GeoGuessrAI\models\train_indices.pkl
Imágenes válidas para entrenar: 135,243 (Se descartaron 4,758,539)


In [7]:
print("\nProcesando dataset de validación")

# Cargar dataset de validación
try:
    val_dataset = load_dataset(
        "osv5m/osv5m", 
        split="test", 
        streaming=False, 
        trust_remote_code=True, 
        keep_in_memory=False
    )
    # Limpieza de columnas igual que en train
    val_dataset = val_dataset.remove_columns([c for c in val_dataset.column_names if c not in {'latitude', 'longitude'}])
    print(f"Dataset validación listo: {len(val_dataset):,} filas.")
except Exception as e:
    print(f"Error cargando validación: {e}")
    raise e

# Calcular tokens para validación
print(f"Generando tokens para validación")
val_results = val_dataset.map(
    batch_process_s2,
    batched=True,
    batch_size=5000,
    num_proc=NUM_CORES,
    remove_columns=['latitude', 'longitude'],
    desc="Tokens Validación"
)

# Filtrar y guardar índices
print("Filtrando imágenes de validación...")
val_indices = []

# Utilizamos valid_cells que calculamos en el split de entrenamiento
if 'valid_cells' not in locals():
    # Si por alguna razón no está en memoria, recargamos
    with open(OUTPUT_FILE, "rb") as f:
        class_data = pickle.load(f)
    valid_cells = set(class_data["cell_to_idx"].keys())

for idx, token in enumerate(tqdm(val_results['s2_token'], desc="Seleccionando Val")):
    if token in valid_cells:
        val_indices.append(idx)

VAL_INDICES_FILE = os.path.join(MODELS_DIR, "val_indices.pkl")
with open(VAL_INDICES_FILE, "wb") as f:
    pickle.dump(val_indices, f)

print(f"¡Éxito! Indices válidos guardados en: {os.path.abspath(VAL_INDICES_FILE)}")
print(f"Imágenes de validación aptas: {len(val_indices):,} (Se descartaron {len(val_results) - len(val_indices):,})")


Procesando dataset de validación
OSV5M {'full': False, 'name': 'osv5m', 'hash': '0ac7fc681aace0f00245c6dea7848bcb0e424845081905154c1adb5f9bf8f19e', 'base_path': 'https://huggingface.co/datasets/osv5m/osv5m/resolve/main', 'token': None, 'use_auth_token': None, 'repo_id': 'osv5m/osv5m', 'storage_options': {}, 'dataset_name': 'osv5m', '_writer_batch_size': None, 'config_kwargs': {}, 'config': BuilderConfig(name='default', version=0.0.0, data_dir=None, data_files=None, description=None), 'config_id': 'default', 'info': DatasetInfo(description='', citation='', homepage='', license='', features={'image': Image(mode=None, decode=True, id=None), 'latitude': Value(dtype='float32', id=None), 'longitude': Value(dtype='float32', id=None), 'country': Value(dtype='string', id=None), 'region': Value(dtype='string', id=None), 'sub-region': Value(dtype='string', id=None), 'city': Value(dtype='string', id=None)}, post_processed=None, supervised_keys=None, task_templates=None, builder_name='osv5m', data

Seleccionando Val: 100%|██████████| 210122/210122 [00:00<00:00, 2022525.63it/s]

¡Éxito! Indices válidos guardados en: d:\Projects\GeoGuessrAI\models\val_indices.pkl
Imágenes de validación aptas: 0 (Se descartaron 210,122)
